# BirdCLEF 2026 — Model_5 + HGNet b4 ONNX (v20, private LB 0.946)

我的银牌方案之一 (private LB **0.946**). HGNet 分支 = **单 backbone hgnetv2_b4** 的
4-fold ONNX ensemble, 以 **15%** 权重 rank-blend 进 Model_5 的 `subm_5.csv`:

- Perch v2 + Distilled-SED 主干 (Model_5) 占 **0.85**
- HGNet b4 分支占 **0.15** (`HGNET_W=0.15`)

走 Model_5 内部 `USE_HGNET` 单 backbone 流程 (留空 `BIRDCLEF_HGNET_ENSEMBLE_DIRS`,
所以 hgnet_addon 走 `HGNET_CKPT_DIR + HGNET_BACKBONE` 老逻辑).

## Required Datasets to Attach

| Dataset | Why |
|---|---|
| `birdclef-2026` (competition) | hidden test audio + sample_submission |
| 你的 **code dataset** (含 `scheme_model_5/` + 顶层 `hgnet_branch.py`) | Model_5 主代码 + HGNet 分支逻辑 |
| 你的 **HGNet b4 ONNX** dataset (含 `best_model_fold0..3.onnx`) | HGNet b4 4-fold |
| `rishikeshjani/perch-onnx-for-birdclef-2026` | Perch ONNX |
| `google/bird-vocalization-classifier` (`tensorflow2/perch_v2_cpu/1`) | Perch TF fallback |
| `ashok205/tf-wheels` | offline TF wheel (Internet OFF) |
| `tuckerarrants/bc2026-distilled-sed-public` | Distilled-SED `sed_fold*.onnx` (或你自己训的 hgnetv2_b2 SED) |

**Settings**: Internet **OFF**, Accelerator: **None** (CPU). Save Version → **Save & Run All** → Submit.

In [ ]:
import os, sys

# =============================================================================
# 0. 用户必填 (按 Kaggle 上 dataset 的实际路径填)
# =============================================================================
# 0a. code dataset 顶层 (含 scheme_model_5/ + hgnet_branch.py)
CODE_DIR     = '/kaggle/input/datasets/czastu/bird-2026-release-inference'

# 0b. HGNet b4 ONNX 目录 (含 best_model_fold0..3.onnx)
#     attach 你的 b4 onnx dataset, 然后从 Kaggle 右边 Input 列表复制实际路径来.
HGNET_B4_DIR = '/kaggle/input/datasets/czastu/hgnetv2-b4-fold-onnx'

# 0c. SED ONNX 目录 (sed_fold*.onnx). 留空 = 让代码 rglob 自动找.
SED_DIR      = ''

# 0d. b4 在最终 rank-blend 中的权重 (★ 0.946 方案 = 0.15, 主干占 0.85)
HGNET_W      = '0.15'

# =============================================================================
# 1. !!! 关键: 必须在 import scheme_model_5 之前设 env vars !!!
#    config.py 是模块级 os.environ.get, import 时一次性读完, 之后改无效.
# =============================================================================
os.environ['BIRDCLEF_MODEL5_SUBM']       = '/kaggle/working/submission.csv'
os.environ['BIRDCLEF_MODE']              = 'submit'
if SED_DIR:
    os.environ['BIRDCLEF_SED_DIR']       = SED_DIR

# HGNet b4 单 backbone 分支 (走内部 USE_HGNET 流程)
#   注意: 不设 BIRDCLEF_HGNET_ENSEMBLE_DIRS -> hgnet_addon 走单 backbone 老逻辑
os.environ['BIRDCLEF_M5_USE_HGNET']      = '1'
os.environ['BIRDCLEF_HGNET_BACKBONE']    = 'hgnetv2_b4.ssld_stage2_ft_in1k'
os.environ['BIRDCLEF_HGNET_DIR']         = HGNET_B4_DIR
os.environ['BIRDCLEF_HGNET_W']           = HGNET_W
os.environ['BIRDCLEF_HGNET_BRANCH_PATH'] = os.path.join(CODE_DIR, 'hgnet_branch.py')

print('>>> env vars set (before import scheme_model_5):')
for _k in ['BIRDCLEF_MODEL5_SUBM', 'BIRDCLEF_M5_USE_HGNET', 'BIRDCLEF_HGNET_BACKBONE',
          'BIRDCLEF_HGNET_DIR', 'BIRDCLEF_HGNET_W', 'BIRDCLEF_HGNET_BRANCH_PATH']:
    print(f'      {_k} = {os.environ.get(_k)!r}')

# =============================================================================
# 2. 校验路径 + sys.path
# =============================================================================
assert os.path.isdir(os.path.join(CODE_DIR, 'scheme_model_5')), \
    f'CODE_DIR={CODE_DIR} 下没有 scheme_model_5/'
_main_py = os.path.join(CODE_DIR, 'scheme_model_5', 'main.py')
with open(_main_py) as _f:
    _src = _f.read()
print(f'>>> CODE_DIR = {CODE_DIR}')
print(f'>>> main.py  = {_main_py}  ({len(_src)} chars)')

# 容错: HGNET_B4_DIR 写错时, 用 rglob 自动找一下
if not os.path.isdir(HGNET_B4_DIR):
    print(f'!!! HGNET_B4_DIR={HGNET_B4_DIR} 不存在, 尝试 rglob 自动查找...')
    import glob as _glob
    _hits = sorted(_glob.glob('/kaggle/input/**/best_model_fold0.onnx', recursive=True))
    print(f'!!! 找到的 best_model_fold0.onnx: {_hits}')
    if _hits:
        HGNET_B4_DIR = os.path.dirname(_hits[0])
        os.environ['BIRDCLEF_HGNET_DIR'] = HGNET_B4_DIR
        print(f'!!! 自动选用: HGNET_B4_DIR = {HGNET_B4_DIR}')
    else:
        raise FileNotFoundError(
            'HGNet b4 ONNX 找不到. 请 attach 含 best_model_fold0..3.onnx 的 dataset, '
            '并把上面 cell 里 HGNET_B4_DIR 改成实际路径.'
        )

# 把 CODE_DIR 放最前, 强制压过其他可能 attach 的同名 dataset
sys.path = [CODE_DIR] + [p for p in sys.path if p != CODE_DIR]
for _k in [k for k in list(sys.modules.keys()) if k.startswith('scheme_model_')]:
    del sys.modules[_k]

# =============================================================================
# 3. import 后确认实际加载的配置
# =============================================================================
import scheme_model_5.main as _sm5_main
from scheme_model_5.config import (
    USE_HGNET as _USE_HGNET,
    HGNET_BACKBONE as _HGNET_BACKBONE,
    HGNET_CKPT_DIR as _HGNET_CKPT_DIR,
    HGNET_W as _HGNET_W,
    HGNET_ENSEMBLE_DIRS as _ENS_DIRS,
    OUTPUT_SUBM as _OUTPUT_SUBM,
)
print(f'>>> import 实际加载       = {_sm5_main.__file__}')
print(f'>>> config.USE_HGNET      = {_USE_HGNET}')
print(f'>>> config.HGNET_BACKBONE = {_HGNET_BACKBONE}')
print(f'>>> config.HGNET_CKPT_DIR = {_HGNET_CKPT_DIR}')
print(f'>>> config.HGNET_W        = {_HGNET_W}')
print(f'>>> config.ENSEMBLE_DIRS  = {_ENS_DIRS}  (应为空 -> 单 backbone 路径)')
print(f'>>> config.OUTPUT_SUBM    = {_OUTPUT_SUBM}')
assert _USE_HGNET, 'USE_HGNET 还是 False! 检查 BIRDCLEF_M5_USE_HGNET env var.'

os.chdir('/kaggle/working')

# =============================================================================
# 4. run Model_5 (主流水线 + HGNet b4 单 backbone rank-blend 15%)
# =============================================================================
_sm5_main.main()

# =============================================================================
# 5. sanity check
# =============================================================================
import pandas as pd
_final_csv = '/kaggle/working/submission.csv'
if not os.path.exists(_final_csv):
    _alt = '/kaggle/working/subm_5.csv'
    if os.path.exists(_alt):
        import shutil
        shutil.copy(_alt, _final_csv)
        print(f'>>> 从 {_alt} 复制到 {_final_csv}')
df = pd.read_csv(_final_csv)
print('submission.csv shape:', df.shape)
print('row_id sample :', df['row_id'].head(3).tolist())
print('value range  :', float(df.iloc[:, 1:].min().min()), '..', float(df.iloc[:, 1:].max().max()))
df.head(2)